# Tensor与数据搬运

## 概述

本节介绍pyasc中的核心数据结构——`GlobalTensor`和`LocalTensor`，以及数据搬运接口`asc.data_copy`。这两类Tensor分别对应AI Core的外部存储和内部存储，数据搬运接口负责在两者之间搬运数据，是连接全局存储和计算单元的桥梁。

### 学习目标

1. 掌握`asc.GlobalTensor`的创建和`set_global_buffer`使用；
2. 掌握`asc.LocalTensor`的创建和`TPosition`逻辑位置；
3. 掌握`asc.data_copy`数据搬运接口和Tensor切片操作；
4. 理解CopyIn→Compute→CopyOut三级数据流。


In [ ]:
# 环境初始化
!mkdir -p Sources/02.04

import os, subprocess
env = subprocess.check_output("bash -l -c 'source $ASCEND_TOOLKIT_HOME/set_env.sh && env'", shell=True, text=True)
for line in env.splitlines():
    if "=" in line: os.environ.__setitem__(*line.split("=", 1))
print("环境初始化完成")

---
# 1. 存储层级与数据流

AI Core的存储分为两级，数据在两级之间搬运形成标准的三级流水：

<img src="./images/data_transfer_flow.png" alt="GM与UB数据搬运流程" width="700px">


<table style="margin-left: 0; margin-right: auto; width: auto; border: 1px solid #ddd;">
  <thead>
    <tr>
      <th align="left">存储层级</th>
      <th align="left">名称</th>
      <th align="left">说明</th>
      <th align="left">对应Tensor</th>
    </tr>
  </thead>
  <tbody>
    <tr><td>外部存储</td><td>Global Memory (HBM)</td><td>Host分配的Device内存，容量大但访问延迟高</td><td><code>asc.GlobalTensor</code></td></tr>
    <tr><td>内部存储</td><td>Unified Buffer (UB)</td><td>AI Core内部高速缓存，Vector算子的Local Memory</td><td><code>asc.LocalTensor</code></td></tr>
  </tbody>
</table>

> **说明：** 本章介绍的是Vector（矢量）类算子的存储层级和数据流，此时Local Memory即Unified Buffer（UB）。对于Cube（矩阵）类算子，Local Memory还包括L1、L0A、L0B和L0C等存储位置，将在第4章Matmul算子开发中介绍。

数据流：`GM(输入) → UB(VECIN) → 计算单元 → UB(VECOUT) → GM(输出)`


---
# 2. GlobalTensor

`asc.GlobalTensor`用来存放**Global Memory**的全局数据。在核函数中，通过`set_global_buffer`设置GlobalTensor的全局内存地址。

```python
# 创建GlobalTensor
x_gm = asc.GlobalTensor()
y_gm = asc.GlobalTensor()
z_gm = asc.GlobalTensor()

# 设置Global Memory起始地址和长度（偏移offset个元素）
x_gm.set_global_buffer(x + offset, block_length)
y_gm.set_global_buffer(y + offset, block_length)
z_gm.set_global_buffer(z + offset, block_length)
```

其中`x`是核函数的`asc.GlobalAddress`参数，`x + offset`表示从起始地址偏移offset个元素。


---
# 3. LocalTensor

`asc.LocalTensor`用于存放AI Core内部存储的数据。对于Vector算子，LocalTensor存放在Unified Buffer（UB）上。LocalTensor需要指定逻辑位置（TPosition）：

<table style="margin-left: 0; margin-right: auto; width: auto; border: 1px solid #ddd;">
  <thead>
    <tr>
      <th align="left">TPosition</th>
      <th align="left">说明</th>
      <th align="left">数据流角色</th>
    </tr>
  </thead>
  <tbody>
    <tr><td><code>VECIN</code></td><td>矢量计算输入位置</td><td>CopyIn目标，存放搬入的输入数据</td></tr>
    <tr><td><code>VECOUT</code></td><td>矢量计算输出位置</td><td>CopyOut源，存放待搬出的结果</td></tr>
    <tr><td><code>VECCALC</code></td><td>矢量计算中间结果位置</td><td>融合算子中间数据</td></tr>
  </tbody>
</table>

在手动同步模式下（如Add样例），LocalTensor直接创建并指定逻辑位置和地址：

```python
# 获取数据类型信息
data_type = x.dtype
buffer_size = tile_length * BUFFER_NUM * data_type.sizeof()

# 创建LocalTensor（数据类型, 逻辑位置, 偏移地址, 长度）
x_local = asc.LocalTensor(data_type, asc.TPosition.VECIN, 0, tile_length * BUFFER_NUM)
y_local = asc.LocalTensor(data_type, asc.TPosition.VECIN, buffer_size, tile_length * BUFFER_NUM)
z_local = asc.LocalTensor(data_type, asc.TPosition.VECOUT, buffer_size * 2, tile_length * BUFFER_NUM)
```

### 数据类型

pyasc中使用`asc.float32`、`asc.float16`、`asc.int32`等表示数据类型，通过`dtype.sizeof()`获取字节大小：

<table style="margin-left: 0; margin-right: auto; width: auto; border: 1px solid #ddd;">
  <thead>
    <tr>
      <th align="left">pyasc数据类型</th>
      <th align="left">对应C++类型</th>
      <th align="left">字节大小</th>
    </tr>
  </thead>
  <tbody>
    <tr><td><code>asc.float32</code></td><td><code>float</code></td><td>4</td></tr>
    <tr><td><code>asc.float16</code></td><td><code>half</code></td><td>2</td></tr>
    <tr><td><code>asc.int32</code></td><td><code>int32_t</code></td><td>4</td></tr>
    <tr><td><code>asc.int16</code></td><td><code>int16_t</code></td><td>2</td></tr>
    <tr><td><code>asc.uint8</code></td><td><code>uint8_t</code></td><td>1</td></tr>
  </tbody>
</table>


> **UB内存约束：** Unified Buffer的容量有限（如Ascend910B上为192KB），算子开发时需要确保所有LocalTensor的总大小不超过UB容量。计算公式：`总UB占用 = sum(各LocalTensor的 tile_length * BUFFER_NUM * dtype.sizeof())`。


---
# 4. data_copy数据搬运

`asc.data_copy`用于在Global Memory和Local Memory之间搬运数据，对应Ascend C的`DataCopy`接口。

### 4.1 基本用法

```python
# CopyIn: GM → LM
asc.data_copy(x_local[buf_id * tile_length:], x_gm[i * tile_length:], tile_length)

# CopyOut: LM → GM
asc.data_copy(z_gm[i * tile_length:], z_local[buf_id * tile_length:], tile_length)
```

第三个参数`tile_length`表示搬运的元素个数。

### 4.2 Tensor切片操作

pyasc支持Tensor切片操作，使用`tensor[offset:]`语法获取从偏移位置开始的子Tensor：

<table style="margin-left: 0; margin-right: auto; width: auto; border: 1px solid #ddd;">
  <thead>
    <tr>
      <th align="left">表达式</th>
      <th align="left">含义</th>
    </tr>
  </thead>
  <tbody>
    <tr><td><code>x_gm[i * tile_length:]</code></td><td>从x_gm的第i*tile_length个元素开始</td></tr>
    <tr><td><code>x_local[buf_id * tile_length:]</code></td><td>从x_local的第buf_id*tile_length个元素开始</td></tr>
  </tbody>
</table>

下图展示了Tensor切片操作在数据搬运中的实际映射关系：

<img src="./images/tensor_slicing.png" alt="Tensor切片操作" width="700px">


---
# 5. 完整示例：Add算子的数据搬运

让我们查看Add样例中完整的Tensor使用和数据搬运流程：

In [ ]:
# 查看Add样例中的Tensor和数据搬运代码
!cat ./src/add.py

### 5.1 Add样例数据搬运分析

<table style="margin-left: 0; margin-right: auto; width: auto; border: 1px solid #ddd;">
  <thead>
    <tr>
      <th align="left">步骤</th>
      <th align="left">操作</th>
      <th align="left">关键接口</th>
      <th align="left">数据流</th>
    </tr>
  </thead>
  <tbody>
    <tr><td>1</td><td>创建GlobalTensor</td><td><code>set_global_buffer</code></td><td>绑定x, y, z的GM地址</td></tr>
    <tr><td>2</td><td>创建LocalTensor</td><td><code>asc.LocalTensor</code></td><td>在VECIN/VECOUT分配LM</td></tr>
    <tr><td>3</td><td>CopyIn</td><td><code>asc.data_copy</code></td><td>GM → LM(VECIN)</td></tr>
    <tr><td>4</td><td>Compute</td><td><code>asc.add</code></td><td>LM(VECIN) → LM(VECOUT)</td></tr>
    <tr><td>5</td><td>CopyOut</td><td><code>asc.data_copy</code></td><td>LM(VECOUT) → GM</td></tr>
  </tbody>
</table>

这个CopyIn→Compute→CopyOut的三级流水是矢量算子的标准编程范式，与Ascend C完全一致。


---
# 6. 小结

- `asc.GlobalTensor`管理Global Memory，通过`set_global_buffer`绑定地址
- `asc.LocalTensor`管理Local Memory，需指定TPosition（VECIN/VECOUT/VECCALC）
- `asc.data_copy`实现GM和LM之间的数据搬运，第三个参数为元素个数
- Tensor切片`tensor[offset:]`获取从偏移位置开始的子Tensor
- 标准数据流：GM → LM(VECIN) → 计算 → LM(VECOUT) → GM


---

## 课后实践

**实践任务：** 查看Add样例代码，回答以下问题：

1. Add样例中，x_local和y_local放在哪个TPosition位置？z_local放在哪个位置？
2. `asc.data_copy`的第三个参数`tile_length`表示什么？
3. `x + offset`中的`x`是什么类型的参数？`offset`是如何计算的？

<details>
<summary>点击查看参考答案</summary>

1. x_local和y_local放在`asc.TPosition.VECIN`（矢量输入位置），z_local放在`asc.TPosition.VECOUT`（矢量输出位置）。
2. `tile_length`表示搬运的元素个数。
3. `x`是核函数的`asc.GlobalAddress`类型参数，指向Global Memory中的输入数据。`offset = asc.get_block_idx() * block_length`，根据当前核索引计算数据偏移。

</details>